# Demo24 endpoint data audit

This reproducible audit checks endpoint-only structural consistency, split membership, mapping, and provenance for the frozen 24-candidate review queue. It does not read TS/IRC geometries, energies, labels, or barriers and does not constitute chemical plausibility review or acceptance. All 24 chemistry reviews remain pending.

The local Jupyter kernel could not be launched because Windows denied `SetFileSecurity` on its connection file. The code cells below were run sequentially in the active project Python interpreter and their text outputs were saved here.

In [1]:
from pathlib import Path
import json
from scripts.audit_demo24_endpoints import audit_demo24

audit = audit_demo24()
report_path = Path("data/manifests/demo24_endpoint_audit_v1.json")
report_path.write_text(json.dumps(audit, ensure_ascii=False, indent=2) + "\n", encoding="utf-8")
print(json.dumps({key: audit[key] for key in ("audit_status", "n_candidates", "split_counts", "review_status_counts", "n_issue_rows", "collection_issues")}, ensure_ascii=False, indent=2))

{
  "audit_status": "consistent_pending_human_review",
  "n_candidates": 24,
  "split_counts": {
    "train": 16,
    "valid": 8
  },
  "review_status_counts": {
    "pending": 24
  },
  "n_issue_rows": 0,
  "collection_issues": []
}


In [2]:
import pandas as pd

review = pd.DataFrame([{
    "reaction_id": row["reaction_id"],
    "split": row["split"],
    "stratum": row["stratum"],
    "mapping_status": row["mapping_status"],
    "issues": ";".join(row["issues"]) or "none",
    "chemistry_review": row["human_review_status"],
} for row in audit["records"]])
print(review.to_string(index=False))

   reaction_id split              stratum              mapping_status issues chemistry_review
RXN_0000007104 train         A_H_transfer             resolved_unique   none          pending
RXN_0000161724 train         A_H_transfer             resolved_unique   none          pending
RXN_0000091050 valid         A_H_transfer             resolved_unique   none          pending
RXN_0000077619 train           B_one_bond             resolved_unique   none          pending
RXN_0000026256 train           B_one_bond             resolved_unique   none          pending
RXN_0000079731 valid           B_one_bond resolved_symmetry_collapsed   none          pending
RXN_0000053132 train       C_substitution             resolved_unique   none          pending
RXN_0000155302 train       C_substitution             resolved_unique   none          pending
RXN_0000171675 valid       C_substitution             resolved_unique   none          pending
RXN_0000100071 train   D_two_bond_coupled             resolv

In [3]:
summary = {
    "all_integrity_checks_passed": all(count == audit["n_candidates"] for count in audit["checks_passed_for_all_candidates"].values()),
    "check_counts": audit["checks_passed_for_all_candidates"],
    "boundary": audit["information_boundary"],
}
print(json.dumps(summary, ensure_ascii=False, indent=2))

{
  "all_integrity_checks_passed": true,
  "check_counts": {
    "atom_identity_match": 24,
    "charge_and_radical_summaries_match": 24,
    "component_counts_match": 24,
    "edit_counts_match": 24,
    "endpoint_geometry_valid": 24,
    "hydrogen_event_counts_match": 24,
    "mapped_bond_edits_match": 24,
    "mapping_status_resolved": 24,
    "no_forbidden_truth_fields": 24,
    "official_split_match": 24,
    "reaction_smiles_parse_and_map_match": 24,
    "review_queue_match": 24,
    "sanitized_export_present": 24
  },
  "boundary": "Endpoint-only structural consistency checks. No TS/IRC fields, geometries, energies, or labels were read. This is not a chemical plausibility review or acceptance."
}
